# 1. What you'll learn

        - compute global and local tree SHAP values
- draw a held-out partial-dependence curve
- duplicate a correlated feature and watch attribution move

        **The one question this notebook answers:** What do SHAP and partial dependence explain—and how can correlated features make an apparently precise story arbitrary?

# 2. Setup

This lesson keeps shap and partial dependence small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import shap

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A seeded tabular outcome depends nonlinearly on age-like risk, income-like capacity, utilization, and an interaction. Ground truth is known, while a duplicated risk measure creates attribution ambiguity later.

**Small example:** If baseline log-odds is -1 and three SHAP contributions are +.8,+.3,-.2, the case log-odds is -.1. Contributions add to the model output, not necessarily to causal effects.

In [ ]:
rng=np.random.default_rng(SEED);n=2200;risk=rng.normal(size=n);capacity=rng.normal(size=n);util=rng.uniform(0,1,n);tenure=rng.exponential(2,n);noise=rng.normal(size=(n,4));logit=-1+1.2*risk-.7*capacity+2*(util>.72)+1.1*risk*util-.15*tenure;prob=1/(1+np.exp(-logit));y=rng.binomial(1,prob);X=pd.DataFrame({"risk":risk,"capacity":capacity,"utilization":util,"tenure":tenure,**{f"noise_{j}":noise[:,j] for j in range(4)}});print("Shape:",X.shape,"positive share:",round(y.mean(),3));print(X.head(3).round(3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].hist(X.risk,bins=25,color="#176b66");axes[0].set_title("Risk distribution");axes[1].scatter(X.utilization,prob,s=8,alpha=.3);axes[1].set_title("Known nonlinear probability")
sns.heatmap(X.corr(),cmap="vlag",center=0,xticklabels=False,yticklabels=False,ax=axes[2]);axes[2].set_title("Original feature correlation");plt.tight_layout();plt.show()

**Takeaway:** Risk spans both protective and harmful values.

**Takeaway:** Utilization has a threshold and interacts with risk, so one coefficient is inadequate.

**Takeaway:** Original predictors are mostly independent, making the first attribution relatively identifiable.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        Training turns the assumptions behind shap and partial dependence into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Fit a tree ensemble on training rows.
2. For SHAP, compare each case against coalitions of present/absent features.
3. Average absolute contributions for global importance.
4. For partial dependence, vary one feature on held-out rows and average predictions.

In [ ]:
model=XGBClassifier(n_estimators=180,max_depth=3,learning_rate=.05,subsample=.8,colsample_bytree=.8,eval_metric="logloss",random_state=SEED,n_jobs=1).fit(X_train,y_train);explainer=shap.TreeExplainer(model);sv=np.asarray(explainer.shap_values(X_test));print("SHAP matrix:",sv.shape,"test AUC:",round(roc_auc_score(y_test,model.predict_proba(X_test)[:,1]),3));print("Case 0 contributions:",pd.Series(sv[0],index=X.columns).sort_values(key=abs,ascending=False).head().round(3).to_dict())

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
importance=pd.Series(np.abs(sv).mean(0),index=X.columns).sort_values();grid=np.linspace(X_test.utilization.quantile(.02),X_test.utilization.quantile(.98),30);pd_values=[]
for value in grid:
 changed=X_test.copy();changed["utilization"]=value;pd_values.append(model.predict_proba(changed)[:,1].mean())
base=np.repeat(y_train.mean(),len(y_test));auc=roc_auc_score(y_test,model.predict_proba(X_test)[:,1]);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].barh(importance.index,importance.values,color="#176b66");axes[0].set_title("Mean |SHAP|")
axes[1].plot(grid,pd_values,color="#d38b45");axes[1].set(title="Partial dependence",xlabel="utilization",ylabel="mean prediction")
case=pd.Series(sv[0],index=X.columns).sort_values();axes[2].barh(case.index,case.values,color=np.where(case.values>0,"#d38b45","#176b66"));axes[2].set_title("One local model explanation");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Model AUC={auc:.3f}; random ranking=.500")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Correlated or duplicated features can divide or exchange SHAP credit while predictions barely change; attribution is not causal ownership. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
Xd=X.copy();Xd["risk_duplicate"]=Xd.risk+rng.normal(0,.01,n);Xa,Xb,ya,yb=train_test_split(Xd,y,test_size=.25,stratify=y,random_state=SEED);dup=XGBClassifier(n_estimators=180,max_depth=3,learning_rate=.05,random_state=SEED,n_jobs=1).fit(Xa,ya);dsv=np.asarray(shap.TreeExplainer(dup).shap_values(Xb));dimp=pd.Series(np.abs(dsv).mean(0),index=Xd.columns)
print(f"Original AUC={auc:.3f}; duplicate-feature AUC={roc_auc_score(yb,dup.predict_proba(Xb)[:,1]):.3f}");print("Attribution original risk:",round(float(importance["risk"]),3),"after duplication split:",round(float(dimp["risk"]),3),"+",round(float(dimp["risk_duplicate"]),3))

**Use this when…** you need to inspect a fitted model's associations, debug cases, and communicate predictive behaviour with caveats.

        **Don't use this when…** you need causal effects, correlated-feature ownership, or explanations detached from a validated model and background distribution.

        ## Try this

        1. Use probability versus log-odds SHAP output.
2. Compute partial dependence inside risk subgroups.
3. Change the SHAP background sample and compare local values.